# Logistic Regression Assumptions – Practice Skeleton

**Short name (GitHub):** `LogReg_Assumptions`  
**Lab source:** Codecademy *Assumptions of Logistic Regression I & II* + sklearn implementation, prediction thresholds, ROC/AUC, class imbalance.  
**Data:** Wisconsin Diagnostic Breast Cancer (`data/breast_cancer_data.csv`, 569 fine-needle aspirates).

Work this notebook first. Peek at `LogReg_Assumptions_Solution.ipynb` only after you have tried the cell.  
Helpers live in `LogReg_Assumptions.py`. Charts of the intended outcome: `lr_assump_flowchart.png`.

> Clinical framing: **missing a malignant tumour (false negative) is far costlier than a false alarm.** Recall is the headline metric; accuracy is secondary because classes are uneven (357 benign / 212 malignant).


## Inline cheat-sheet (keep this cell visible)

See also **`LogReg_Assumptions_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Encode target | `df['diagnosis'].map({'M':1,'B':0})` |
| Binary check | `df.diagnosis.value_counts()` — exactly two classes |
| Independence | `df.id.nunique() == df.id.count()` |
| Events-per-variable | `max_features = min(class counts) / 10` |
| Logit linearity | `sns.regplot(..., logistic=True)` should look sigmoidal |
| Multicollinearity | `sns.heatmap(X.corr())` — drop one of a pair with \|r\| ≳ 0.8 |
| Unregularized LR | `LogisticRegression(penalty=None, fit_intercept=True)` *(old sklearn: `penalty='none'`)* |
| Soft scores | `predict_proba(X)[:, 1]` |
| Hard class at *t* | `(proba >= t).astype(int)` |
| Recall (TPR) | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$ |
| Precision | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FP})$ |
| FPR | $\mathrm{FP}/(\mathrm{FP}+\mathrm{TN})$ |
| ROC | `roc_curve(y, proba)` → (FPR, TPR, thresholds) |
| AUC | `roc_auc_score(y, proba)` — 0.5 = chance, 1 = perfect |
| Stratify | `train_test_split(..., stratify=y)` |
| Balanced weights | `class_weight='balanced'` → $w_c \propto 1/n_c$ |

Default threshold $t=0.5$ is a *convention*, not a law. Raising *t* ↓FP ↑FN; lowering *t* ↑FP ↓FN.


## 0. Packages

Run this cell first. `penalty=None` requires scikit-learn ≥ 1.2.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score,
)
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler

%matplotlib inline
sns.set_style("whitegrid")
np.set_printoptions(precision=4, suppress=True)
print("libraries ready")


## 1. Load and encode

### Task 1.1
Read `data/breast_cancer_data.csv`. Drop any `Unnamed` column. Map diagnosis `M→1`, `B→0`. Print `head()` and `diagnosis.value_counts()`.


In [ ]:
# YOUR CODE HERE
df = None

print(df.head())
print(df.diagnosis.value_counts())


## 2. Assumptions I — target, independence, sample size, outliers

The four primary assumptions from the lesson:

1. **Binary target**
2. **Independent observations** (no repeated patients)
3. **Large enough sample** for MLE (rule of thumb: ≥ 10 events per feature in the *smaller* class)
4. **No influential outliers** (LR coefficients are sensitive to extremes)

### Task 2.1 — binary target
Print the distinct diagnosis values and their frequencies (already done if 1.1 is correct). Confirm there are exactly two classes.


In [ ]:
# YOUR CODE HERE
print(df.diagnosis.value_counts())
print("n classes:", df.diagnosis.nunique())


### Task 2.2 — independent observations
Test whether the number of unique `id` values equals the number of rows.


In [ ]:
# YOUR CODE HERE
unique_ids = None
print(unique_ids)


### Task 2.3 — events-per-variable cap
`max_features = min(class counts) / 10`. Print it. With 212 malignant cases the cap is 21.2 — our 4-feature working set is comfortably inside.


In [ ]:
# YOUR CODE HERE
max_features = None
print(max_features)


### Task 2.4 — outlier screen on the *mean* features
Features are right-skewed and strictly positive, so a raw boxplot is hard to read. Plot the **log-then-z-score** boxplot:

```python
sns.boxplot(data=np.log(df[predictor_all] + 0.01).apply(zscore))
```

Which feature shows the most extreme upper tail?


In [ ]:
predictor_all = [
    "radius_mean", "texture_mean", "perimeter_mean", "area_mean",
    "smoothness_mean", "compactness_mean", "concavity_mean",
    "concave points_mean", "symmetry_mean", "fractal_dimension_mean",
]
# YOUR CODE HERE


### Task 2.5 — drop extreme `fractal_dimension_mean`
Keep rows below the 99th percentile. Save as `df_filtered`. How many rows were removed?


In [ ]:
# YOUR CODE HERE
q_hi = None
df_filtered = None
print("q99 =", q_hi, " rows kept =", len(df_filtered), " dropped =", len(df) - len(df_filtered))


### Task 2.6 — redraw the boxplot on `df_filtered`
Does the fractal-dimension tail calm down? (Other features are unchanged.)


In [ ]:
# YOUR CODE HERE


## 3. Assumptions II — logit linearity and multicollinearity

### Task 3.1 — visual logit-linearity
`sns.regplot(x=feature, y='diagnosis', data=df, logistic=True)` should look like a sigmoid if the feature is linearly related to the log-odds.

Compare `radius_mean` (should look sigmoidal) with `fractal_dimension_mean` (flatter / weaker).


In [ ]:
# YOUR CODE HERE


### Task 3.2 — correlation heatmap
Two features are almost collinear with `radius_mean` (geometry: perimeter and area are functions of radius). Name them.

There is another highly correlated pair among the remaining features. Store it as `correlated_pair`.


In [ ]:
x = df[predictor_all]
# YOUR CODE HERE — heatmap
correlated_pair = []  # e.g. ['compactness_mean', 'concavity_mean']
print("correlated_pair:", correlated_pair)


## 4. scikit-learn implementation

Working feature set (drops the collinear size trio down to radius only):

`predictor_var = ['radius_mean', 'texture_mean', 'compactness_mean', 'symmetry_mean']`

### Task 4.1 — construct the estimator
Unregularized, with intercept. Print `.get_params()`.


In [ ]:
predictor_var = ["radius_mean", "texture_mean", "compactness_mean", "symmetry_mean"]
outcome_var = "diagnosis"
x_train, x_test, y_train, y_test = train_test_split(
    df[predictor_var], df[outcome_var], random_state=0, test_size=0.3
)
# YOUR CODE HERE
log_reg = None
print(log_reg.get_params())


### Task 4.2 — fit and read coefficients
Store `coefficients` and `intercept`. A large positive coefficient on compactness means a bump in that feature sharply raises log-odds of malignancy.


In [ ]:
# YOUR CODE HERE
coefficients = None
intercept = None
print("coefficients:", coefficients)
print("intercept:", intercept)


### Task 4.3 — test-set metrics

$$
\text{accuracy}=\frac{TP+TN}{n},\quad
\text{precision}=\frac{TP}{TP+FP},\quad
\text{recall}=\frac{TP}{TP+FN},\quad
F_1=2\cdot\frac{P\cdot R}{P+R}
$$

Which metric is highest / lowest on this split?


In [ ]:
# YOUR CODE HERE
y_pred = None
accuracy = None
precision = None
recall = None
f1 = None
print(f"accuracy\t{accuracy}")
print(f"precision\t{precision}")
print(f"recall   \t{recall}")
print(f"f1       \t{f1}")


### Task 4.4 — confusion matrix as a labelled DataFrame
How many tumours were predicted correctly? How many were predicted benign but were actually malignant (FN — the dangerous cell)?


In [ ]:
# YOUR CODE HERE
test_conf_matrix = None
print(test_conf_matrix)


## 5. Prediction thresholds

`predict` uses $t=0.5$. `predict_proba` returns `[P(y=0), P(y=1)]`.

### Task 5.1 — rebuild the 0.5 class from probabilities
Confirm it matches `y_pred` with `np.array_equal`.


In [ ]:
y_pred_prob = log_reg.predict_proba(x_test)
# YOUR CODE HERE
y_pred_class = None
diff = None
print("same as predict()?", diff)


### Task 5.2 / 5.3 — confusion matrices at 25%, 50%, 75%
As *t* rises you should see FN ↑ and FP ↓.


In [ ]:
# YOUR CODE HERE
print("CM 50%"); print(None)
print("CM 25%"); print(None)
print("CM 75%"); print(None)


### Task 5.4 — clinical threshold
Sweep `thresh = np.linspace(0, 1, 100)`. Record FN at each *t*. What is the **lowest** threshold at which FN first reaches 2? Store as `thresh_choice`.

(Interpretation: below that *t* you have 0–1 misses; crossing it you start allowing 2 misses.)


In [ ]:
thresh = np.linspace(0, 1, 100)
false_negatives = []
# YOUR CODE HERE
thresh_choice = None
print("thresh_choice =", thresh_choice)


## 6. ROC curve and AUC

### Task 6.1
Plot the ROC of your model and of a `DummyClassifier(strategy='most_frequent')`. Label a few thresholds on the curve.


In [ ]:
# YOUR CODE HERE


### Task 6.2
Compute `roc_auc` from `y_test` and the **positive-class probability**. Values near 1 mean the two classes are well separated by the score.


In [ ]:
# YOUR CODE HERE
roc_auc = None
print("ROC AUC:", roc_auc)


## 7. Class imbalance

Positivity rate $= n_{\text{malignant}} / n$. Overall it is $212/569 \approx 0.37$.

### Task 7.1 — stratified split
`random_state=6`, `test_size=0.3`, `stratify=df[outcome_var]`.


In [ ]:
x_train_u, x_test_u, y_train_u, y_test_u = train_test_split(
    df[predictor_var], df[outcome_var], random_state=6, test_size=0.3
)
print("unstrat train pos", y_train_u.mean(), "test pos", y_test_u.mean())

# YOUR CODE HERE — stratified names: x_train_str, x_test_str, y_train_str, y_test_str


### Task 7.2 — positivity rates after stratification
They should nearly match each other (and the population rate).


In [ ]:
# YOUR CODE HERE
str_train_positivity_rate = None
str_test_positivity_rate = None
print(str_train_positivity_rate, str_test_positivity_rate)


### Task 7.3 — recall / accuracy on the stratified test fold


In [ ]:
# YOUR CODE HERE
recall_str = None
accuracy_str = None
print("stratified recall, acc:", recall_str, accuracy_str)


### Task 7.4 / 7.5 — `class_weight='balanced'`
Fit on the *unstratified* `random_state=6` split so you can compare three regimes: default, stratified, balanced-weights. Print recall and accuracy for the balanced model.


In [ ]:
# YOUR CODE HERE
log_reg_bal = None
recall_bal = None
accuracy_bal = None
print("balanced recall, acc:", recall_bal, accuracy_bal)


## 8. Alternate code (same scientific result)

### Task 8.1 — statsmodels Logit
`sm.Logit(y, sm.add_constant(X)).fit()`. Compare coefficient signs/magnitudes with sklearn. statsmodels gives SEs and a Wald table “for free”.


In [ ]:
# YOUR CODE HERE
# import statsmodels.api as sm


### Task 8.2 — scale-then-LR pipeline
`StandardScaler` + the same unregularized estimator. Predictions should be nearly identical (affine features + unpenalized LR). Coefficients will *look* different because they are in z-units.


In [ ]:
# YOUR CODE HERE


### Task 8.3 — NumPy threshold helper
Write `predict_at(proba, t)` in one line and reuse it for a 9-point sweep.


In [ ]:
def predict_at(proba, t=0.5):
    # YOUR CODE HERE
    pass

for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pred = predict_at(y_pred_prob[:, 1], t)
    print(t, "FN", confusion_matrix(y_test, pred)[1, 0],
          "FP", confusion_matrix(y_test, pred)[0, 1])


## 9. More practice

### Task 9.1 — swap in the *worst* (largest) nucleus features
`radius_worst, texture_worst, compactness_worst, symmetry_worst`. Refit, print recall and AUC. Do the “worst” measurements separate classes better than the means?


In [ ]:
worst_var = ["radius_worst", "texture_worst", "compactness_worst", "symmetry_worst"]
# YOUR CODE HERE


### Task 9.2 — transfer problem: loan default
`data/lr_assump_loans.csv` has `dti`, `utilization`, `income`, `default`.

1. Check positivity rate and the 10-EPV cap.
2. Fit unregularized LR, print coefficients.
3. Report test recall / AUC.
4. Choose a threshold that keeps FN ≤ 3 on the test fold (or document that you cannot).


In [ ]:
loans = pd.read_csv("data/lr_assump_loans.csv")
# YOUR CODE HERE


### Task 9.3 — which metric when?
Write one sentence for each scenario:

* screening every mammogram in a healthy population
* confirming a biopsy already flagged as suspicious
* a balanced A/B test of two email subject lines


In [ ]:
screening = """..."""
confirming = """..."""
ab_test = """..."""
print(screening); print(confirming); print(ab_test)


## 10. Simulation (edit the parameters)

Change the boxed values and re-run. The cell:

1. Draws `N_REPS` stratified train/test splits of size `N` (sampled from the 569 rows, with replacement if `N>569`).
2. Optionally flips a fraction `NOISE` of training labels.
3. Fits default vs `class_weight='balanced'` LR.
4. Records test recall, accuracy, and AUC at threshold `T`.
5. Plots boxplots.

Questions to answer after you poke the knobs:

* What happens to recall if you raise `T` from 0.5 to 0.8?
* Does `class_weight='balanced'` still help when `N=80`?
* How much label noise (`NOISE=0.1`) does it take to drag AUC under 0.90?


In [ ]:
# --- editable parameters ---
N = 569          # sample size drawn from the official 569 rows
N_REPS = 20
NOISE = 0.00     # training-label flip probability
T = 0.50         # decision threshold
TEST_SIZE = 0.30
SEED = 0
# ---------------------------

rng = np.random.default_rng(SEED)
rows_def, rows_bal = [], []
pool_X = df[predictor_var].to_numpy()
pool_y = df[outcome_var].to_numpy()

for r in range(N_REPS):
    idx = rng.choice(len(df), size=N, replace=(N > len(df)))
    X = pool_X[idx]; y = pool_y[idx]
    try:
        Xtr, Xte, ytr, yte = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=r, stratify=y
        )
    except ValueError:
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=r)
    if NOISE > 0:
        flip = rng.random(len(ytr)) < NOISE
        ytr = ytr.copy(); ytr[flip] = 1 - ytr[flip]
    for tag, cw in (("default", None), ("balanced", "balanced")):
        m = LogisticRegression(penalty=None, fit_intercept=True, max_iter=4000, class_weight=cw)
        m.fit(Xtr, ytr)
        proba = m.predict_proba(Xte)[:, 1]
        pred = (proba >= T).astype(int)
        rec = dict(
            recall=recall_score(yte, pred, zero_division=0),
            accuracy=accuracy_score(yte, pred),
            auc=roc_auc_score(yte, proba) if len(np.unique(yte)) == 2 else np.nan,
        )
        (rows_def if tag == "default" else rows_bal).append(rec)

sim_def = pd.DataFrame(rows_def); sim_bal = pd.DataFrame(rows_bal)
print("DEFAULT\n", sim_def.agg(["mean", "std"]).round(3))
print("BALANCED\n", sim_bal.agg(["mean", "std"]).round(3))

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, col in zip(axes, ["recall", "accuracy", "auc"]):
    ax.boxplot([sim_def[col].dropna(), sim_bal[col].dropna()], labels=["default", "balanced"])
    ax.set_title(col); ax.set_ylim(0.4, 1.02)
fig.suptitle(f"N={N}  T={T}  noise={NOISE}  reps={N_REPS}", y=1.03)
plt.tight_layout(); plt.show()


## 11. Audience rewrite (from the attached PDFs)

Using Jočys (data literacy, subject knowledge) and McMurrey (expert / technician / executive / nonspecialist), rewrite **one finding** — e.g. “at *t*=0.25 the model misses 2 malignancies and raises 14 false alarms; AUC ≈ 0.98” — four ways. Keep each paragraph ≤ 80 words.


In [ ]:
expert = """..."""
technician = """..."""
executive = """..."""
nonspecialist = """..."""
print("EXPERT\n", expert)
print("TECHNICIAN\n", technician)
print("EXECUTIVE\n", executive)
print("NONSPECIALIST\n", nonspecialist)


## 12. Top-10 applications — and when **not** to use this procedure

List 10 settings where *binary logistic regression + assumption checks + threshold/ROC work* is the right tool, and 6–8 settings where it is the wrong tool (multiclass already, rare events with <10 EPV, clustered patients, non-linear logit, causal claims from observational features, etc.). One line each.


In [ ]:
applications = [
    # "1. ...",
]
not_appropriate = [
    # "1. ...",
]
for row in applications: print(row)
print("--- not appropriate ---")
for row in not_appropriate: print(row)


## 13. What you should have when you finish

- [ ] Encoded binary target, unique-id check, 10-EPV cap printed
- [ ] Outlier boxplots before / after the 99th-percentile filter
- [ ] Two logistic `regplot`s and a labelled correlation heatmap
- [ ] Unregularized sklearn model with coefficients, 4 metrics, confusion matrix
- [ ] Threshold CMs at 0.25 / 0.50 / 0.75 and a `thresh_choice` for FN = 2
- [ ] ROC vs dummy + AUC
- [ ] Stratified positivity rates and a `class_weight='balanced'` comparison
- [ ] At least one alternate stack (statsmodels or scaled pipeline)
- [ ] Worst-feature and loan-default practice
- [ ] Simulation that moves when you edit `N`, `T`, `NOISE`
- [ ] Four audience paragraphs + applications / anti-applications list

**Companion files:** `LogReg_Assumptions_Solution.ipynb`, `LogReg_Assumptions_Reusable_Template.ipynb`, `LogReg_Assumptions.py`, `LogReg_Assumptions_Cheatsheet.docx`, `LogReg_Assumptions_1Page_Summary_Report.docx`, `LogReg_Assumptions_Project_Memo.docx`, `LogReg_Assumptions_Strategy_Guide.docx`, `lr_assump_flowchart.png`.
